<a href="https://colab.research.google.com/github/samridhi-gupta08/Project-Minerva/blob/main/Samridhi_250968_BCS_Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import json

url = "https://www.unwomen.org/en/articles/faqs/faqs-afghanistan"

r = requests.get(url)

soup = BeautifulSoup(r.text, "html.parser")

title = soup.find("h1").text.strip()

paras = soup.find_all("p")

text = ""

for p in paras:
    t = p.get_text().strip()

    if t != "":
        text += t + "\n"

print(title)
print(text[:1000])

with open("article.txt", "w", encoding="utf-8") as f:
    f.write(text)

data = {
    "title": title,
    "content": text
}

with open("article.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

Error 403 Forbidden
Forbidden
Details: cache-bur-kbur8200055-BUR 1779470981 2142745155
Varnish cache server



In [2]:
import nltk
import spacy
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

with open("article.txt", "r", encoding="utf-8") as f:
    text = f.read()

sentences = sent_tokenize(text)

words = word_tokenize(text)

print("sentences:", len(sentences))
print("words:", len(words))

print(sentences[:3])
print(words[:20])

nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

filtered = []

for token in doc:
    if not token.is_stop and not token.is_punct:
        filtered.append(token.text)

print(filtered[:40])

lemmas = []

for token in doc:
    lemmas.append(token.lemma_)

print(lemmas[:40])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


sentences: 1
words: 9
['Forbidden\nDetails: cache-bur-kbur8200055-BUR 1779470981 2142745155\nVarnish cache server']
['Forbidden', 'Details', ':', 'cache-bur-kbur8200055-BUR', '1779470981', '2142745155', 'Varnish', 'cache', 'server']
['Forbidden', '\n', 'Details', 'cache', 'bur', 'kbur8200055', 'BUR', '1779470981', '2142745155', '\n', 'Varnish', 'cache', 'server', '\n']
['Forbidden', '\n', 'detail', ':', 'cache', '-', 'bur', '-', 'kbur8200055', '-', 'bur', '1779470981', '2142745155', '\n', 'varnish', 'cache', 'server', '\n']


In [3]:
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

with open("article.txt", "r", encoding="utf-8") as f:
    text = f.read()

sents = sent_tokenize(text)

chunks = []

size = 5

for i in range(0, len(sents), size):
    chunk = " ".join(sents[i:i+size])
    chunks.append(chunk)

print("chunks:", len(chunks))

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print(embeddings.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


chunks: 1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(1, 384)


In [4]:
!pip install chromadb
!pip install rfc3987
import chromadb
import pickle

from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt")

with open("article.txt", "r", encoding="utf-8") as f:
    text = f.read()

sents = sent_tokenize(text)

parts = []

for i in range(0, len(sents), 5):
    parts.append(" ".join(sents[i:i+5]))

model = SentenceTransformer("all-MiniLM-L6-v2")

vecs = model.encode(parts)

client = chromadb.Client()

collection = client.create_collection("afghan_data")

collection.add(
    documents=parts,
    embeddings=vecs.tolist(),
    ids=[str(i) for i in range(len(parts))]
)

print("done")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


done


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()

collection = client.get_collection("afghan_data")

q = "women not allowed to study"

q_embed = model.encode([q]).tolist()

result = collection.query(
    query_embeddings=q_embed,
    n_results=2
)

print(result["documents"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[['Forbidden\nDetails: cache-bur-kbur8200055-BUR 1779470981 2142745155\nVarnish cache server']]


In [9]:
#requirements.txt
requests
beautifulsoup4
nltk
spacy
sentence-transformers
chromadb

NameError: name 'beautifulsoup4' is not defined